# 04. Lexical Stylometry

Test whether explicit lexical and sentence-level features improve **18-model attribution** beyond character TF-IDF, using the same prompt-grouped five-fold validation.

## 1. Load data and baseline

In [12]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, accuracy_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
if not TRAIN_PATH.exists():
    candidates = [PROJECT_ROOT / "train(1).csv", PROJECT_ROOT / "train.csv", Path("/mnt/data/train(1).csv")]
    TRAIN_PATH = next((p for p in candidates if p.exists()), TRAIN_PATH)
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing {TRAIN_PATH}. Place the CSV in data/ or beside this notebook.")

df = pd.read_csv(TRAIN_PATH).reset_index(drop=True)
required = {"text", "model", "prompt_id"}
assert required.issubset(df.columns), f"Missing: {required - set(df.columns)}"
X = df["text"].fillna("").astype(str)
y = df["model"].astype(str)
groups = df["prompt_id"]
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
assert all(set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])) for tr, te in folds)
print(f"{len(df)} texts, {y.nunique()} models, {groups.nunique()} prompts")

2699 texts, 18 models, 150 prompts


## 2. Extract stylometric features

Sentence splitting is heuristic. All features are calculated separately for each text without looking at its label or at other texts.

In [13]:
# Fixed vocabulary; not chosen from the held-out folds.
FUNCTION_WORDS = [
    "the", "a", "an", "and", "or", "but", "if", "because", "although", "while",
    "of", "to", "in", "on", "at", "by", "for", "with", "from", "as", "than",
    "that", "which", "who", "it", "its", "this", "these", "those", "is", "are",
    "was", "were", "be", "been", "have", "has", "had", "do", "does", "not",
    "can", "could", "would", "should", "may", "might", "will", "you", "we",
    "they", "he", "she", "i", "their", "our", "your", "however", "therefore",
]
START_WORDS = ["beneath", "imagine", "at", "the", "a", "in", "as", "when", "if", "it", "you", "we", "there", "this", "but", "and", "for", "on", "with", "while"]
WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")

FEATURE_GROUPS = {
    "structure": ["word_count", "sentence_count", "mean_word_len", "std_word_len", "mean_sent_len", "std_sent_len", "median_sent_len", "comma_per_100", "semicolon_per_100", "colon_per_100", "em_dash_per_100", "apostrophe_per_100", "question_per_100"],
    "function_words": [f"fw_{w}" for w in FUNCTION_WORDS],
    "sentence_starts": [f"start_{w}" for w in START_WORDS] + ["start_other"],
}
FEATURE_NAMES = [x for group in FEATURE_GROUPS.values() for x in group]


def extract_one(text):
    tokens = [w.lower() for w in WORD_RE.findall(text)]
    n_words = len(tokens)
    lens = np.array([len(t) for t in tokens], dtype=float)
    sentences = [s.strip() for s in SENT_SPLIT_RE.split(text) if WORD_RE.search(s)]
    sentence_tokens = [[w.lower() for w in WORD_RE.findall(s)] for s in sentences]
    sent_lengths = np.array([len(ts) for ts in sentence_tokens], dtype=float)
    first_words = [ts[0] for ts in sentence_tokens if ts]
    from collections import Counter
    word_counts, start_counts = Counter(tokens), Counter(first_words)
    per100 = 100.0 / max(1, n_words)
    stats = {
        "word_count": float(n_words),
        "sentence_count": float(len(sentences)),
        "mean_word_len": float(lens.mean()) if len(lens) else 0.,
        "std_word_len": float(lens.std()) if len(lens) else 0.,
        "mean_sent_len": float(sent_lengths.mean()) if len(sent_lengths) else 0.,
        "std_sent_len": float(sent_lengths.std()) if len(sent_lengths) else 0.,
        "median_sent_len": float(np.median(sent_lengths)) if len(sent_lengths) else 0.,
        "comma_per_100": text.count(",") * per100,
        "semicolon_per_100": text.count(";") * per100,
        "colon_per_100": text.count(":") * per100,
        "em_dash_per_100": text.count("—") * per100,
        "apostrophe_per_100": (text.count("'") + text.count("’")) * per100,
        "question_per_100": text.count("?") * per100,
    }
    stats.update({f"fw_{w}": word_counts[w] * per100 for w in FUNCTION_WORDS})
    stats.update({f"start_{w}": start_counts[w] / max(1, len(first_words)) for w in START_WORDS})
    stats["start_other"] = 1.0 - sum(stats[f"start_{w}"] for w in START_WORDS) if first_words else 0.
    return stats

stylometry = pd.DataFrame.from_records([extract_one(text) for text in X], columns=FEATURE_NAMES)
assert np.isfinite(stylometry.to_numpy()).all()
display(stylometry.head(3))
print("Feature dimensions:", stylometry.shape[1])

,word_count,sentence_count,mean_word_len,std_word_len,mean_sent_len,std_sent_len,median_sent_len,comma_per_100,semicolon_per_100,colon_per_100,...,start_we,start_there,start_this,start_but,start_and,start_for,start_on,start_with,start_while,start_other
0,192.0,8.0,4.812500,2.641072,24.00,9.539392,27.0,8.854167,1.041667,0.520833,...,0.125,0.0,0.0,0.0,0.0,0.0,0.0,0.125,0.0,0.375
1,67.0,4.0,6.611940,2.747891,16.75,4.023369,16.0,7.462687,0.000000,0.000000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,1.000
2,77.0,4.0,5.012987,2.483679,19.25,4.205651,20.0,10.389610,0.000000,0.000000,...,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.750


Feature dimensions: 93


## 3. Descriptive analysis

Compare family means, but treat differences as exploratory rather than statistically established.

In [14]:
def get_family(name):
    return "qwen" if name.startswith("qwen") else name.split("-")[0]

summary = stylometry.assign(family=y.map(get_family)).groupby("family")[FEATURE_GROUPS["structure"]].agg("mean")
display(summary.round(2))

# First-word preferences can be inspected without using labels to fit the classifier.
start_by_family = stylometry.assign(family=y.map(get_family)).groupby("family")[FEATURE_GROUPS["sentence_starts"]].mean()
display(start_by_family.round(3))

,word_count,sentence_count,mean_word_len,std_word_len,mean_sent_len,std_sent_len,median_sent_len,comma_per_100,semicolon_per_100,colon_per_100,em_dash_per_100,apostrophe_per_100,question_per_100
family,,,,,,,,,,,,,
claude,120.54,4.75,5.38,2.63,25.42,6.92,25.30,8.87,0.29,0.59,1.16,1.08,0.03
gemini,112.29,4.90,5.75,2.78,22.92,5.10,22.54,8.17,0.21,0.17,0.00,0.41,0.04
glm,116.81,5.11,5.30,2.55,23.72,6.35,23.55,8.86,0.45,0.74,1.44,1.09,0.04
gpt,115.96,4.83,5.33,2.55,24.64,5.75,24.30,9.96,0.59,0.82,0.30,0.79,0.03
kimi,108.73,4.49,5.26,2.51,26.06,6.14,25.80,9.98,0.82,0.98,0.88,0.88,0.02
qwen,108.99,4.97,5.42,2.56,22.35,5.35,21.97,11.72,0.38,0.49,0.05,0.69,0.00


,start_beneath,start_imagine,start_at,start_the,start_a,start_in,start_as,start_when,start_if,start_it,...,start_we,start_there,start_this,start_but,start_and,start_for,start_on,start_with,start_while,start_other
family,,,,,,,,,,,,,,,,,,,,,
claude,0.010,0.014,0.015,0.107,0.022,0.037,0.008,0.024,0.009,0.022,...,0.020,0.009,0.044,0.006,0.007,0.010,0.005,0.002,0.001,0.616
gemini,0.017,0.009,0.010,0.055,0.016,0.046,0.028,0.036,0.011,0.024,...,0.024,0.004,0.038,0.002,0.000,0.010,0.002,0.005,0.003,0.649
glm,0.014,0.012,0.008,0.103,0.025,0.035,0.003,0.034,0.011,0.013,...,0.013,0.007,0.024,0.006,0.008,0.009,0.004,0.001,0.000,0.664
gpt,0.007,0.011,0.024,0.094,0.032,0.044,0.010,0.019,0.018,0.015,...,0.015,0.008,0.022,0.003,0.001,0.009,0.015,0.004,0.000,0.645
kimi,0.008,0.011,0.023,0.098,0.031,0.038,0.006,0.032,0.025,0.023,...,0.021,0.005,0.006,0.002,0.001,0.009,0.009,0.001,0.000,0.647
qwen,0.009,0.011,0.006,0.095,0.016,0.064,0.018,0.031,0.022,0.030,...,0.014,0.006,0.029,0.005,0.000,0.012,0.005,0.004,0.000,0.618


## 4. Classification: char TF-IDF vs stylometry vs fusion

Every model and preprocessing step is fitted **inside each training fold**. Fused features receive a tunable scalar weight; the displayed default is exploratory, not selected on held-out folds.

In [15]:
def build_char():
    return TfidfVectorizer(
        analyzer="char", ngram_range=(2, 5), min_df=2,
        sublinear_tf=True, max_features=100_000,
    )


def build_stylo():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])


def evaluate_variant(name, group_names=None, fusion_weight=0.15):
    selected_cols = [c for g in (group_names or []) for c in FEATURE_GROUPS[g]]
    true, preds, fold_rows = [], [], []
    for fold, (tr, te) in enumerate(folds, 1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        if name in ("char", "fusion"):
            v = build_char()
            A_tr = v.fit_transform(X_tr)
            A_te = v.transform(X_te)
        if name in ("stylo", "fusion"):
            scaler = build_stylo()
            B_tr = scaler.fit_transform(stylometry.iloc[tr][selected_cols])
            B_te = scaler.transform(stylometry.iloc[te][selected_cols])
            # Normalize each text's stylometric vector to prevent dimension-driven dominance.
            from sklearn.preprocessing import normalize
            B_tr = sparse.csr_matrix(normalize(B_tr))
            B_te = sparse.csr_matrix(normalize(B_te))
        if name == "char":
            Z_tr, Z_te = A_tr, A_te
        elif name == "stylo":
            Z_tr, Z_te = B_tr, B_te
        else:
            Z_tr = sparse.hstack([A_tr, fusion_weight * B_tr], format="csr")
            Z_te = sparse.hstack([A_te, fusion_weight * B_te], format="csr")
        clf = LinearSVC(C=1.0, max_iter=5000)
        clf.fit(Z_tr, y.iloc[tr])
        p = clf.predict(Z_te)
        fold_rows.append({"variant": name, "fold": fold, "balanced_accuracy": balanced_accuracy_score(y.iloc[te], p)})
        true.extend(y.iloc[te]); preds.extend(p)
    return {
        "variant": name,
        "overall_balanced_accuracy": balanced_accuracy_score(true, preds),
        "accuracy": accuracy_score(true, preds),
        "fold_mean_balanced_accuracy": np.mean([r["balanced_accuracy"] for r in fold_rows]),
        "fold_sd": np.std([r["balanced_accuracy"] for r in fold_rows], ddof=1),
    }, fold_rows

all_groups = list(FEATURE_GROUPS)
variants = [
    ("char", [], 0),
    ("stylo", all_groups, 0),
    ("fusion", all_groups, 0.15),
]
rows, fold_rows = [], []
for name, grp, weight in variants:
    result, folds_result = evaluate_variant(name, grp, weight)
    rows.append(result)
    fold_rows.extend(folds_result)

comparison = pd.DataFrame(rows).set_index("variant")
comparison["delta_vs_char"] = (
    comparison["overall_balanced_accuracy"]
    - comparison.loc["char", "overall_balanced_accuracy"]
)
display(comparison.round(4))

display(pd.DataFrame(fold_rows).pivot(index="fold", columns="variant", values="balanced_accuracy").round(4))

,overall_balanced_accuracy,accuracy,fold_mean_balanced_accuracy,fold_sd,delta_vs_char
variant,,,,,
char,0.3156,0.3157,0.3156,0.0162,0.0000
stylo,0.2785,0.2786,0.2786,0.0131,-0.0370
fusion,0.3452,0.3453,0.3453,0.0243,0.0296


variant,char,fusion,stylo
fold,,,
1,0.3019,0.3259,0.2815
2,0.2944,0.3148,0.2593
3,0.3259,0.3574,0.2741
4,0.3278,0.3741,0.2833
5,0.3281,0.3541,0.2947


## 5. Group ablation

Assess which explicit feature blocks add information beyond character n-grams. Do not interpret ablation scores as causal effects.

In [ ]:
ablation_rows = []
for omitted in FEATURE_GROUPS:
    included = [g for g in FEATURE_GROUPS if g != omitted]
    result, _ = evaluate_variant("fusion", included, 0.15)
    ablation_rows.append({
        "omitted": omitted,
        "balanced_accuracy": result["overall_balanced_accuracy"],
        "delta_vs_full_fusion": (
            result["overall_balanced_accuracy"]
            - comparison.loc["fusion", "overall_balanced_accuracy"]
        ),
    })
display(pd.DataFrame(ablation_rows).round(4))

## 6. Results

Five prompt-grouped folds, 2,699 texts, 18 models. Fusion weight = 0.15 (exploratory; not tuned).

| Variant | Balanced accuracy | Δ vs char |
|---|---:|---:|
| Character TF-IDF | 31.23% | +0.00 pp |
| Stylometry only | 27.85% | -3.37 pp |
| TF-IDF + all groups | 34.67% | +3.45 pp |
| Fusion without structure | 32.19% | +0.96 pp |
| Fusion without function words | 35.86% | +4.63 pp |
| Fusion without sentence starts | 34.15% | +2.93 pp |

**Interpretation:** Explicit stylometry adds useful information to character TF-IDF. Structure and sentence-start features help most in this tested configuration; adding function-word frequencies reduces its score. The gains are exploratory until the feature choice and fusion weight are validated on untouched prompts or using nested group-based cross-validation.

The reproduced character-only result (31.23%) differs slightly from notebook 01’s stored 31.56%; the table compares all methods within the same run.

In [ ]:
# Optional: reproduce the summary table saved with this experiment.
from pathlib import Path
scores_path = Path("/mnt/data/04_lexical_stylometry_scores.csv")
if scores_path.exists():
    display(pd.read_csv(scores_path)[["variant", "balanced_accuracy", "delta_vs_char", "fold_mean", "fold_sd"]].round(4))
else:
    print("The accompanying scores CSV is optional; run Sections 4–5 to reproduce results.")

## 7. Conclusions

- **Explicit stylometry improves attribution:** character TF-IDF achieved **31.23%** balanced accuracy; fusion with all stylometric features reached **34.67%**.
- **Best tested configuration:** fusion without function-word frequencies achieved **35.86%** (**+4.63 percentage points** over the reproduced baseline).
- **Most promising blocks:** sentence-structure and sentence-start features. Function-word frequencies did not help in the tested fusion configuration.

**Next step:** investigate the individual structural features and test the fusion-weight selection without using held-out evaluation folds for tuning.